# PE posterior violins with CV-bias markers for multiple injection IDs

This notebook extends the single-injection notebook so that several `inj_id` values are loaded and displayed together.

For each single-parameter FTI recovery, it plots grouped posterior violins of

\[
\frac{\theta-\theta_{\rm ref}}{\sigma_{\rm CV}}
\]

and overlays the corresponding Cutler–Vallisneri prediction

\[
\frac{\Delta\theta_{\rm CV}}{\sigma_{\rm CV}}.
\]

The color identifies the injection ID. The notebook creates:

1. one grouped plot for every active FTI recovery;
2. one multipage PDF containing all grouped recovery plots;
3. one grouped summary plot containing only the active FTI parameter from every recovery;
4. CSV tables combining all requested injection IDs.

Edit only `INJ_IDS` in the settings cell to choose the injections to compare.

## 1. Imports

In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

import lisabeta.utils.plotutils as plotutils

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 200)

# Avoid failures caused by an unavailable external LaTeX installation.
plt.rcParams["text.usetex"] = False
plt.rcParams["mathtext.fontset"] = "dejavusans"

## 2. Paths and settings

In [ ]:
# ---------------------------------------------------------------------
# Main settings
# ---------------------------------------------------------------------

# Put all injection IDs that should appear together in this list.
INJ_IDS = [0, 1, 2, 3, 4]

PE_BASE_DIR = Path("/pbs/home/a/amahdi/PE_for_my_work")
CASE_ID_TEMPLATE = "M1e6_SEOBv5_id_{inj_id}"

CV_PATH_CANDIDATES = [
    Path(
        "/pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/"
        "SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_"
        "domega_dtau_scan_M1e6/run_data/bias_syst_recovery/"
        "cv_bias_sigma_fti_all_params_all_inj_ids.csv"
    ),
]

# Optional custom labels. Any injection absent from this dictionary is
# displayed simply as "inj_id = N".
INJECTION_LABELS = {
    # 0: r"$\mathrm{inj\_id}=0$",
    # 1: r"$\mathrm{inj\_id}=1$",
}

inj_slug = "_".join(str(inj_id) for inj_id in INJ_IDS)

OUTPUT_ROOT = (
    PE_BASE_DIR
    / "combined_inj_id_plots"
    / f"inj_ids_{inj_slug}"
)
PER_FTI_DIR = OUTPUT_ROOT / "per_fti_recovery"
TABLE_DIR = OUTPUT_ROOT / "tables"

for directory in [OUTPUT_ROOT, PER_FTI_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Posterior cleaning
# ---------------------------------------------------------------------

BAD_LNL_CUT = -1e12

# This is retained for compatibility with helper functions from the
# previous notebook.
SIGMA_METHOD = "std"

MAX_VIOLIN_SAMPLES = 50000
RANDOM_SEED = 20260717


# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------

VIOLIN_QUANTILES = (0.05, 0.95)

# Set to (-50, 50) to match the range in the supervisor's example.
# Set to None to calculate a common symmetric range automatically.
Y_LIMITS = (-50, 50)

SHOW_PE_MEDIAN = True
SHOW_PLUS_MINUS_ONE = True
SAVE_DPI = 250

# Total horizontal width occupied by all injections at one parameter.
GROUP_TOTAL_WIDTH = 0.78

# "truth": centre each PE posterior on its own injected value.
#
# "reference_table": centre on values supplied in a CSV. For several
# injection IDs, the recommended columns are:
# inj_id, active_fti_run, parameter, reference_value
CENTER_MODE = "truth"
REFERENCE_TABLE_PATH = None


# ---------------------------------------------------------------------
# Parameter order
# ---------------------------------------------------------------------

PHYSICAL_PARAMETER_ORDER = [
    "Mchirp",
    "q",
    "chiPN",
    "chim",
    "Deltat",
    "dist",
    "inc",
    "phi",
    "lambda",
    "beta",
    "psi",
]

FTI_PARAMETER_ORDER = [
    "fti_dchiMinus2v",
    "fti_dxi0v",
    "fti_dchi0v",
    "fti_dchi1v",
    "fti_dchi2v",
    "fti_dchi3v",
    "fti_dchi4v",
    "fti_dchi5lv",
    "fti_dchi6v",
    "fti_dchi6lv",
    "fti_dchi7v",
    "fti_dkappas",
    "fti_dkappaSv",
]


def injection_label(inj_id):
    return INJECTION_LABELS.get(
        inj_id,
        rf"$\mathrm{{inj\_id}}={inj_id}$",
    )


if not INJ_IDS:
    raise ValueError("INJ_IDS must contain at least one injection ID.")

if len(set(INJ_IDS)) != len(INJ_IDS):
    raise ValueError("INJ_IDS contains duplicate values.")

print("Injection IDs:", INJ_IDS)
print("Output directory:", OUTPUT_ROOT)

## 3. Discover configuration files

In [ ]:
config_records = []
missing_config_rows = []

for inj_id in INJ_IDS:
    case_id = CASE_ID_TEMPLATE.format(inj_id=inj_id)
    config_dir = PE_BASE_DIR / "config_files" / case_id
    config_files = sorted(config_dir.glob("*.json"))

    if not config_files:
        missing_config_rows.append({
            "inj_id": inj_id,
            "config_directory": str(config_dir),
            "reason": "No JSON configuration files found",
        })
        print(
            f"[warning] no JSON configuration files found "
            f"for inj_id={inj_id}: {config_dir}"
        )
        continue

    print(
        f"inj_id={inj_id}: found {len(config_files)} "
        f"configuration files in {config_dir}"
    )

    for config_file in config_files:
        config_records.append({
            "inj_id": inj_id,
            "case_id": case_id,
            "config_dir": config_dir,
            "config_file": config_file,
        })

if not config_records:
    raise FileNotFoundError(
        "No configuration files were found for any requested injection ID."
    )

config_index = pd.DataFrame(config_records)
display(config_index.head(30))

if missing_config_rows:
    missing_config_table = pd.DataFrame(missing_config_rows)
    display(missing_config_table)

## 4. Locate and load processed/raw PE files

In [ ]:
def infer_pe_base_dir_from_config_location(config_file):
    config_file = Path(config_file)
    return config_file.parent.parent.parent


def infer_run_data_root_from_config_location(config_file):
    return (
        infer_pe_base_dir_from_config_location(config_file)
        / "run_data"
    )


def infer_run_dir_from_config_location(config_file):
    config_file = Path(config_file)
    return (
        infer_run_data_root_from_config_location(config_file)
        / config_file.parent.name
    )


def unique_labeled_paths(paths):
    unique = []
    seen = set()

    for label, path in paths:
        path = Path(path)
        key = str(path)

        if key not in seen:
            unique.append((label, path))
            seen.add(key)

    return unique


def candidate_output_paths(
    config_file,
    out_dir,
    out_name,
    suffix,
):
    config_file = Path(config_file)
    out_dir = Path(out_dir)

    run_data_root = (
        infer_run_data_root_from_config_location(config_file)
    )
    inferred_run_dir = (
        infer_run_dir_from_config_location(config_file)
    )
    case_id = config_file.parent.name

    candidates = [
        (
            "configured_string_concat",
            Path(str(out_dir) + f"{out_name}{suffix}"),
        ),
        (
            "inferred_string_concat",
            run_data_root / f"{case_id}{out_name}{suffix}",
        ),
        (
            "configured_path_join",
            out_dir / f"{out_name}{suffix}",
        ),
        (
            "inferred_path_join",
            inferred_run_dir / f"{out_name}{suffix}",
        ),
    ]

    return unique_labeled_paths(candidates)


def first_existing_path(candidates):
    for label, path in candidates:
        if path.exists():
            return label, path

    return None, None


def detect_active_fti_parameter(config):
    infer_params = list(
        config.get("prior_params", {})
        .get("infer_params", [])
    )

    configured_fti = (
        config.get("waveform_params", {})
        .get("tgr_params", {})
        .get("FTI", {})
        .get("fti_params", [])
    )

    active = [
        parameter
        for parameter in infer_params
        if parameter in configured_fti
        or str(parameter).startswith("fti_")
    ]

    if active:
        return active[-1]

    return None


def read_case_information(config_file, inj_id):
    config_file = Path(config_file)

    with config_file.open("r") as stream:
        config = json.load(stream)

    run_params = config.get("run_params", {})
    infer_params = list(
        config.get("prior_params", {})
        .get("infer_params", [])
    )

    out_dir = Path(run_params.get("out_dir", ""))
    out_name = run_params.get(
        "out_name",
        config_file.stem,
    )

    processed_candidates = candidate_output_paths(
        config_file,
        out_dir,
        out_name,
        ".h5",
    )
    raw_candidates = candidate_output_paths(
        config_file,
        out_dir,
        out_name,
        "_raw.h5",
    )

    processed_kind, processed_file = (
        first_existing_path(processed_candidates)
    )
    raw_kind, raw_file = (
        first_existing_path(raw_candidates)
    )

    return {
        "inj_id": int(inj_id),
        "config_file": config_file,
        "config": config,
        "run_params": run_params,
        "infer_params": infer_params,
        "active_fti": detect_active_fti_parameter(config),
        "out_dir": out_dir,
        "out_name": out_name,
        "processed_candidates": processed_candidates,
        "raw_candidates": raw_candidates,
        "processed_kind": processed_kind,
        "processed_file": processed_file,
        "raw_kind": raw_kind,
        "raw_file": raw_file,
    }


def load_raw_file(raw_file, infer_params):
    posterior = {}

    with h5py.File(raw_file, "r") as h5:
        for parameter in infer_params:
            if parameter in h5:
                posterior[parameter] = h5[parameter][:]
            else:
                print(
                    f"[warning] {parameter} is absent "
                    f"from {raw_file}"
                )

        if "lnlike" in h5:
            posterior["lnL"] = h5["lnlike"][:]
        elif "lnL" in h5:
            posterior["lnL"] = h5["lnL"][:]
        else:
            raise KeyError(
                f"Neither 'lnlike' nor 'lnL' exists in {raw_file}"
            )

    return posterior


def load_case(config_file, inj_id):
    case = read_case_information(
        config_file=config_file,
        inj_id=inj_id,
    )

    case["processed_posterior"] = None
    case["raw_posterior"] = None

    if case["processed_file"] is not None:
        try:
            case["processed_posterior"] = (
                plotutils.load_params_posterior_lisa_smbh(
                    str(case["config_file"]),
                    str(case["processed_file"]),
                )
            )
        except Exception as error:
            print(
                "[warning] processed posterior could not be loaded "
                f"for inj_id={inj_id}, {case['active_fti']}: {error}"
            )

    if case["raw_file"] is not None:
        try:
            case["raw_posterior"] = load_raw_file(
                case["raw_file"],
                case["infer_params"],
            )
        except Exception as error:
            print(
                "[warning] raw posterior could not be loaded "
                f"for inj_id={inj_id}, {case['active_fti']}: {error}"
            )

    return case

In [ ]:
cases = [
    load_case(
        config_file=record["config_file"],
        inj_id=record["inj_id"],
    )
    for record in config_records
]

case_rows = []

for case in cases:
    usable = (
        case["processed_posterior"] is not None
        or case["raw_posterior"] is not None
    )

    case_rows.append({
        "inj_id": case["inj_id"],
        "active_fti": case["active_fti"],
        "config_file": str(case["config_file"]),
        "processed_file": (
            str(case["processed_file"])
            if case["processed_file"] is not None
            else None
        ),
        "raw_file": (
            str(case["raw_file"])
            if case["raw_file"] is not None
            else None
        ),
        "usable_posterior_loaded": usable,
        "infer_params": ", ".join(case["infer_params"]),
    })

case_table = pd.DataFrame(case_rows).sort_values(
    ["inj_id", "active_fti"],
    na_position="last",
)

display(case_table)

print(
    "Cases with at least one loaded posterior:",
    int(case_table["usable_posterior_loaded"].sum()),
    "/",
    len(case_table),
)

## 5. Posterior extraction

In [ ]:
def processed_has_parameter(case, parameter):
    processed = case.get("processed_posterior")

    if processed is None:
        return False

    try:
        return parameter in processed["post"]
    except Exception:
        return False


def raw_has_parameter(case, parameter):
    raw = case.get("raw_posterior")

    try:
        return raw is not None and parameter in raw
    except Exception:
        return False


def choose_posterior_source(case, parameters):
    active_fti = case["active_fti"]

    processed_available = [
        parameter
        for parameter in parameters
        if processed_has_parameter(case, parameter)
    ]
    raw_available = [
        parameter
        for parameter in parameters
        if raw_has_parameter(case, parameter)
    ]

    if (
        active_fti in processed_available
        and len(processed_available) == len(parameters)
    ):
        return "processed", processed_available

    if active_fti in raw_available:
        missing = [
            parameter
            for parameter in parameters
            if parameter not in raw_available
        ]

        if missing:
            print(
                f"[warning] raw posterior for inj_id={case['inj_id']}, "
                f"{active_fti} is missing {missing}"
            )

        return "raw", raw_available

    if active_fti in processed_available:
        missing = [
            parameter
            for parameter in parameters
            if parameter not in processed_available
        ]

        if missing:
            print(
                f"[warning] processed posterior for "
                f"inj_id={case['inj_id']}, {active_fti} "
                f"is missing {missing}"
            )

        return "processed", processed_available

    print(
        f"[error] active FTI parameter {active_fti} is absent "
        f"for inj_id={case['inj_id']} from both posterior sources"
    )
    return None, []


def decide_raw_burnin(case):
    raw = case.get("raw_posterior")

    if raw is None or "lnL" not in raw:
        return 0, "no raw likelihood"

    number_of_steps = int(
        np.asarray(raw["lnL"]).shape[-1]
    )

    run_params = case.get("run_params", {})

    try:
        n_iter = (
            int(run_params["n_iter"])
            if run_params.get("n_iter") is not None
            else None
        )
    except Exception:
        n_iter = None

    try:
        burn_in = int(
            run_params.get("burn_in", 0) or 0
        )
    except Exception:
        burn_in = 0

    if burn_in <= 0:
        return 0, "burn_in is zero or missing"

    if burn_in >= number_of_steps:
        return 0, (
            f"burn_in={burn_in} is not smaller than "
            f"raw nsteps={number_of_steps}"
        )

    if n_iter is not None and number_of_steps == n_iter:
        return burn_in, (
            f"raw nsteps={number_of_steps} equals "
            f"configured n_iter={n_iter}"
        )

    if (
        n_iter is not None
        and number_of_steps == max(n_iter - burn_in, 0)
    ):
        return 0, (
            "raw posterior already appears to have burn-in removed"
        )

    return 0, (
        "raw-chain length is ambiguous; "
        "no additional burn-in was removed"
    )


def slice_last_axis(array, start=0):
    array = np.asarray(array)
    index = [slice(None)] * array.ndim
    index[-1] = slice(start, None)
    return array[tuple(index)]


def align_array_to_likelihood(array, likelihood):
    array = np.asarray(array)
    likelihood = np.asarray(likelihood)

    if array.shape == likelihood.shape:
        return array

    if array.size == likelihood.size:
        return array.reshape(likelihood.shape)

    return None


def extract_processed_matrix(case, parameters):
    post = case["processed_posterior"]["post"]

    arrays = [
        np.asarray(post[parameter]).reshape(-1)
        for parameter in parameters
    ]

    lengths = [array.size for array in arrays]

    if len(set(lengths)) != 1:
        raise ValueError(
            "Processed posterior parameter lengths differ: "
            f"{dict(zip(parameters, lengths))}"
        )

    finite_mask = np.ones(lengths[0], dtype=bool)

    for array in arrays:
        finite_mask &= np.isfinite(array)

    samples = np.column_stack(
        [array[finite_mask] for array in arrays]
    )

    return samples, {
        "source": "processed",
        "burnin_used": 0,
        "burnin_reason": "processed posterior",
    }


def extract_raw_matrix(case, parameters):
    raw = case["raw_posterior"]
    likelihood_full = np.asarray(raw["lnL"])

    burnin, burnin_reason = decide_raw_burnin(case)

    likelihood = slice_last_axis(
        likelihood_full,
        start=burnin,
    )

    mask = (
        np.isfinite(likelihood)
        & (likelihood > BAD_LNL_CUT)
    )

    arrays = []

    for parameter in parameters:
        aligned = align_array_to_likelihood(
            raw[parameter],
            likelihood_full,
        )

        if aligned is None:
            raise ValueError(
                f"{parameter} with shape "
                f"{np.asarray(raw[parameter]).shape} cannot "
                f"be aligned to likelihood shape "
                f"{likelihood_full.shape}"
            )

        sliced = slice_last_axis(
            aligned,
            start=burnin,
        )

        mask &= np.isfinite(sliced)
        arrays.append(sliced)

    samples = np.column_stack(
        [array[mask] for array in arrays]
    )

    return samples, {
        "source": "raw",
        "burnin_used": burnin,
        "burnin_reason": burnin_reason,
    }


def extract_posterior_matrix(case, source, parameters):
    if source == "processed":
        return extract_processed_matrix(
            case,
            parameters,
        )

    if source == "raw":
        return extract_raw_matrix(
            case,
            parameters,
        )

    raise ValueError(f"Unknown posterior source: {source}")

## 6. Injected/reference values

In [ ]:
def get_true_value(case, parameter):
    processed = case.get("processed_posterior")

    if processed is not None:
        injected = processed.get("injparams_Lframe", {})

        try:
            if parameter in injected:
                value = float(
                    np.asarray(injected[parameter]).squeeze()
                )

                if np.isfinite(value):
                    return value
        except Exception:
            pass

    config = case["config"]
    source_params = config.get("source_params", {})

    try:
        if parameter in source_params:
            value = float(
                np.asarray(source_params[parameter]).squeeze()
            )

            if np.isfinite(value):
                return value
    except Exception:
        pass

    fti_block = (
        config.get("waveform_params", {})
        .get("tgr_params", {})
        .get("FTI", {})
    )

    if (
        parameter == "Mchirp"
        and fti_block.get("Mchirp_inj") is not None
    ):
        return float(fti_block["Mchirp_inj"])

    if (
        parameter == "fti_dxi0v"
        and fti_block.get("dxi0v_inj") is not None
    ):
        return float(fti_block["dxi0v_inj"])

    # Every single-parameter FTI recovery is assumed to recover a
    # GR value of zero for its active deviation parameter.
    if parameter == case.get("active_fti"):
        return 0.0

    return None


reference_table = None

if CENTER_MODE == "reference_table":
    if REFERENCE_TABLE_PATH is None:
        raise ValueError(
            "CENTER_MODE='reference_table' requires "
            "REFERENCE_TABLE_PATH."
        )

    reference_table = pd.read_csv(REFERENCE_TABLE_PATH)

    required = {
        "active_fti_run",
        "parameter",
        "reference_value",
    }
    missing = required.difference(reference_table.columns)

    if missing:
        raise ValueError(
            f"Reference table is missing columns: {sorted(missing)}"
        )

    duplicate_key = ["active_fti_run", "parameter"]

    if "inj_id" in reference_table.columns:
        duplicate_key = [
            "inj_id",
            "active_fti_run",
            "parameter",
        ]

    if reference_table.duplicated(duplicate_key).any():
        raise ValueError(
            f"Duplicate rows exist for key {duplicate_key} "
            "in the reference table."
        )


def get_reference_value(case, parameter):
    if CENTER_MODE == "truth":
        value = get_true_value(case, parameter)

    elif CENTER_MODE == "reference_table":
        rows = reference_table.loc[
            (reference_table["active_fti_run"] == case["active_fti"])
            & (reference_table["parameter"] == parameter)
        ]

        if "inj_id" in reference_table.columns:
            rows = rows.loc[
                rows["inj_id"] == case["inj_id"]
            ]

        if len(rows) != 1:
            raise KeyError(
                f"Expected one reference row for inj_id={case['inj_id']}, "
                f"{case['active_fti']} / {parameter}; "
                f"found {len(rows)}."
            )

        value = float(rows.iloc[0]["reference_value"])

    else:
        raise ValueError(
            "CENTER_MODE must be 'truth' or 'reference_table'."
        )

    if value is None or not np.isfinite(value):
        raise ValueError(
            f"No finite reference value for inj_id={case['inj_id']}, "
            f"{case['active_fti']} / {parameter}."
        )

    return float(value)

## 7. Load the CV/Fisher table

In [ ]:
def resolve_existing_path(candidates, label):
    for candidate in candidates:
        candidate = Path(candidate)

        if candidate.exists():
            print(f"{label}: {candidate.resolve()}")
            return candidate

    checked = "\n".join(
        f"  - {Path(path)}"
        for path in candidates
    )

    raise FileNotFoundError(
        f"Could not find {label}. Paths checked:\n{checked}"
    )


CV_PATH = resolve_existing_path(
    CV_PATH_CANDIDATES,
    "CV-bias table",
)

cv_all = pd.read_csv(CV_PATH)

required_cv_columns = {
    "inj_id",
    "fti_case",
    "parameter",
    "Delta_theta_over_sigma",
    "Delta_theta",
    "sigma",
}

missing = required_cv_columns.difference(cv_all.columns)

if missing:
    raise ValueError(
        f"CV table is missing columns: {sorted(missing)}"
    )

cv = cv_all.loc[
    cv_all["inj_id"].isin(INJ_IDS)
].copy()

if cv.empty:
    raise ValueError(
        f"No CV rows were found for INJ_IDS={INJ_IDS}."
    )

duplicate_key = [
    "inj_id",
    "fti_case",
    "parameter",
]

if cv.duplicated(duplicate_key).any():
    duplicates = cv.loc[
        cv.duplicated(
            duplicate_key,
            keep=False,
        )
    ].sort_values(duplicate_key)

    display(duplicates)
    raise ValueError(
        "Duplicate inj_id/fti_case/parameter rows were found."
    )

cv_lookup = cv.set_index(
    duplicate_key
).sort_index()

print("CV rows retained:", len(cv))
print("CV injection IDs:", sorted(cv["inj_id"].unique()))
print("CV FTI cases:", cv["fti_case"].nunique())
display(
    cv.groupby(["inj_id", "fti_case"])
    .size()
    .rename("number_of_parameters")
    .reset_index()
)

## 8. Parameter labels and ordering

In [ ]:
PHYSICAL_LABELS = {
    "Mchirp": r"$\mathcal{M}_c$",
    "q": r"$q$",
    "chiPN": r"$\chi_{\rm PN}$",
    "chim": r"$\chi_{-}$",
    "Deltat": r"$\Delta t$",
    "dist": r"$d_L$",
    "inc": r"$\iota$",
    "phi": r"$\phi$",
    "lambda": r"$\lambda$",
    "beta": r"$\beta$",
    "psi": r"$\psi$",
}


def parameter_label(parameter):
    if parameter in PHYSICAL_LABELS:
        return PHYSICAL_LABELS[parameter]

    name = str(parameter)

    if name == "fti_dxi0v":
        return r"$\delta\hat{\xi}_{0}$"

    if "kappa" in name.lower():
        return r"$\delta\kappa_s$"

    match = re.fullmatch(
        r"fti_dchi(?P<order>Minus\d+|\d+)(?P<log>l?)v",
        name,
    )

    if match:
        order = match.group("order")

        if order.startswith("Minus"):
            order = "-" + order.removeprefix("Minus")

        log_suffix = r"\ell" if match.group("log") else ""

        return (
            rf"$\delta\hat{{\varphi}}_"
            rf"{{{order}{log_suffix}}}$"
        )

    safe = name.replace("_", r"\_")
    return rf"$\mathrm{{{safe}}}$"


def ordered_parameters_for_case(case):
    inferred = list(dict.fromkeys(case["infer_params"]))
    active_fti = case["active_fti"]

    ordered = [
        parameter
        for parameter in PHYSICAL_PARAMETER_ORDER
        if parameter in inferred
    ]

    remaining = [
        parameter
        for parameter in inferred
        if parameter not in ordered
        and parameter != active_fti
        and not str(parameter).startswith("fti_")
    ]
    ordered.extend(remaining)

    if active_fti is not None and active_fti in inferred:
        ordered.append(active_fti)

    return ordered


def combined_parameter_order(entries):
    all_parameters = {
        parameter
        for entry in entries
        for parameter in entry["parameters"]
    }

    ordered = [
        parameter
        for parameter in PHYSICAL_PARAMETER_ORDER
        if parameter in all_parameters
    ]

    non_fti_remaining = sorted(
        parameter
        for parameter in all_parameters
        if parameter not in ordered
        and not str(parameter).startswith("fti_")
    )
    ordered.extend(non_fti_remaining)

    active_fti = entries[0]["active_fti"]

    other_fti = [
        parameter
        for parameter in FTI_PARAMETER_ORDER
        if parameter in all_parameters
        and parameter != active_fti
    ]
    ordered.extend(other_fti)

    unknown_fti = sorted(
        parameter
        for parameter in all_parameters
        if str(parameter).startswith("fti_")
        and parameter not in ordered
        and parameter != active_fti
    )
    ordered.extend(unknown_fti)

    if active_fti in all_parameters:
        ordered.append(active_fti)

    return ordered


def ordered_active_fti_parameters(parameters):
    parameters = list(dict.fromkeys(parameters))

    ordered = [
        parameter
        for parameter in FTI_PARAMETER_ORDER
        if parameter in parameters
    ]

    ordered.extend(
        sorted(
            parameter
            for parameter in parameters
            if parameter not in ordered
        )
    )

    return ordered


def downsample_values(values, maximum, seed):
    values = np.asarray(values)

    if maximum is None or len(values) <= maximum:
        return values

    rng = np.random.default_rng(seed)
    indices = rng.choice(
        len(values),
        size=maximum,
        replace=False,
    )

    return values[indices]

## 9. Build normalized posterior arrays for all injection IDs

In [ ]:
plot_cases = []
summary_rows = []
skipped_rows = []

for case_index, case in enumerate(cases):
    inj_id = case["inj_id"]
    active_fti = case["active_fti"]

    if active_fti is None:
        skipped_rows.append({
            "inj_id": inj_id,
            "config_file": str(case["config_file"]),
            "reason": "No active FTI parameter detected",
        })
        continue

    parameters = ordered_parameters_for_case(case)

    if not parameters:
        skipped_rows.append({
            "inj_id": inj_id,
            "config_file": str(case["config_file"]),
            "active_fti_run": active_fti,
            "reason": "No parameters selected",
        })
        continue

    try:
        source, available_parameters = choose_posterior_source(
            case,
            parameters,
        )

        if source is None:
            raise ValueError(
                "Neither processed nor raw posterior contains "
                "the active FTI parameter."
            )

        selected_parameters = [
            parameter
            for parameter in parameters
            if parameter in available_parameters
        ]

        sample_matrix, source_metadata = extract_posterior_matrix(
            case,
            source,
            selected_parameters,
        )

    except Exception as error:
        skipped_rows.append({
            "inj_id": inj_id,
            "config_file": str(case["config_file"]),
            "active_fti_run": active_fti,
            "reason": str(error),
        })
        continue

    normalized_full = []
    normalized_plot = []
    retained_parameters = []
    cv_points = []
    pe_medians = []

    for parameter_index, parameter in enumerate(
        selected_parameters
    ):
        cv_key = (inj_id, active_fti, parameter)

        if cv_key not in cv_lookup.index:
            print(
                f"[warning] no CV row for inj_id={inj_id}, "
                f"{active_fti} / {parameter}; skipped."
            )
            continue

        cv_row = cv_lookup.loc[cv_key]

        sigma_cv = float(cv_row["sigma"])
        delta_theta_cv = float(cv_row["Delta_theta"])

        if not np.isfinite(sigma_cv) or sigma_cv <= 0:
            print(
                f"[warning] invalid sigma_CV={sigma_cv} for "
                f"inj_id={inj_id}, {active_fti} / {parameter}; "
                "skipped."
            )
            continue

        try:
            reference_value = get_reference_value(
                case,
                parameter,
            )
        except Exception as error:
            print(
                f"[warning] reference-value failure for "
                f"inj_id={inj_id}, {active_fti} / {parameter}: "
                f"{error}"
            )
            continue

        samples = np.asarray(
            sample_matrix[:, parameter_index],
            dtype=float,
        )
        samples = samples[np.isfinite(samples)]

        if samples.size < 2:
            print(
                f"[warning] fewer than two samples for "
                f"inj_id={inj_id}, {active_fti} / {parameter}; "
                "skipped."
            )
            continue

        normalized = (
            samples - reference_value
        ) / sigma_cv

        cv_point = delta_theta_cv / sigma_cv
        stored_cv_point = float(
            cv_row["Delta_theta_over_sigma"]
        )

        if not np.isclose(
            cv_point,
            stored_cv_point,
            rtol=1e-7,
            atol=1e-10,
        ):
            print(
                f"[warning] recomputed and stored CV ratios differ "
                f"for inj_id={inj_id}, "
                f"{active_fti} / {parameter}."
            )

        plot_values = downsample_values(
            normalized,
            MAX_VIOLIN_SAMPLES,
            RANDOM_SEED
            + 100000 * int(inj_id)
            + 1000 * case_index
            + parameter_index,
        )

        q05, q16, median, q84, q95 = np.quantile(
            normalized,
            [0.05, 0.16, 0.50, 0.84, 0.95],
        )

        retained_parameters.append(parameter)
        normalized_full.append(normalized)
        normalized_plot.append(plot_values)
        cv_points.append(cv_point)
        pe_medians.append(median)

        summary_rows.append({
            "inj_id": inj_id,
            "active_fti_run": active_fti,
            "parameter": parameter,
            "posterior_source": source,
            "reference_mode": CENTER_MODE,
            "reference_value": reference_value,
            "sigma_CV": sigma_cv,
            "CV_delta_theta": delta_theta_cv,
            "CV_bias_over_sigma": cv_point,
            "number_of_posterior_samples": int(len(normalized)),
            "PE_mean_over_sigma_CV": float(np.mean(normalized)),
            "PE_median_over_sigma_CV": float(median),
            "PE_q05_over_sigma_CV": float(q05),
            "PE_q16_over_sigma_CV": float(q16),
            "PE_q84_over_sigma_CV": float(q84),
            "PE_q95_over_sigma_CV": float(q95),
            "PE_median_minus_CV": float(median - cv_point),
            "raw_burnin_used": source_metadata["burnin_used"],
            "raw_burnin_reason": source_metadata["burnin_reason"],
            "config_file": str(case["config_file"]),
        })

    if retained_parameters:
        plot_cases.append({
            "case": case,
            "inj_id": inj_id,
            "active_fti": active_fti,
            "parameters": retained_parameters,
            "labels": [
                parameter_label(parameter)
                for parameter in retained_parameters
            ],
            "normalized_full": normalized_full,
            "normalized_plot": normalized_plot,
            "cv_points": np.asarray(cv_points),
            "pe_medians": np.asarray(pe_medians),
            "posterior_source": source,
        })


summary_table = pd.DataFrame(summary_rows)

if summary_table.empty:
    raise ValueError(
        "No usable PE/CV combinations were built. "
        "Inspect the warnings above."
    )

summary_path = (
    TABLE_DIR
    / f"posterior_violin_summary_inj_ids_{inj_slug}.csv"
)
summary_table.to_csv(summary_path, index=False)

print("Usable injection/recovery cases:", len(plot_cases))
print("Summary rows:", len(summary_table))
print("Saved:", summary_path)

availability = (
    summary_table.groupby(
        ["inj_id", "active_fti_run"]
    )["parameter"]
    .nunique()
    .rename("number_of_parameters")
    .reset_index()
)
display(availability)

if skipped_rows or missing_config_rows:
    all_skipped_rows = (
        list(missing_config_rows)
        + list(skipped_rows)
    )
    skipped_table = pd.DataFrame(all_skipped_rows)
    skipped_path = (
        TABLE_DIR
        / f"skipped_cases_inj_ids_{inj_slug}.csv"
    )
    skipped_table.to_csv(skipped_path, index=False)
    display(skipped_table)
    print("Saved:", skipped_path)

display(summary_table.head(40))

## 10. Grouped plotting functions

In [ ]:
available_inj_ids = [
    inj_id
    for inj_id in INJ_IDS
    if any(
        plot_case["inj_id"] == inj_id
        for plot_case in plot_cases
    )
]

if not available_inj_ids:
    raise ValueError(
        "No requested injection ID has usable plotting data."
    )

base_cycle = plt.rcParams[
    "axes.prop_cycle"
].by_key()["color"]

INJ_COLORS = {
    inj_id: base_cycle[index % len(base_cycle)]
    for index, inj_id in enumerate(available_inj_ids)
}


def automatic_y_limits(sample_lists, cv_points):
    values = []

    for samples in sample_lists:
        samples = np.asarray(samples, dtype=float)
        samples = samples[np.isfinite(samples)]

        if samples.size:
            values.extend(
                np.quantile(
                    samples,
                    [0.005, 0.995],
                ).tolist()
            )

    finite_cv = np.asarray(cv_points, dtype=float)
    values.extend(
        finite_cv[np.isfinite(finite_cv)].tolist()
    )

    if not values:
        return (-1.0, 1.0)

    maximum = max(
        abs(min(values)),
        abs(max(values)),
    )
    maximum = max(1.0, 1.1 * maximum)

    return (-maximum, maximum)


def grouped_offsets(inj_ids):
    number = len(inj_ids)

    if number == 1:
        return {inj_ids[0]: 0.0}, GROUP_TOTAL_WIDTH * 0.75

    slot_width = GROUP_TOTAL_WIDTH / number
    offsets = (
        np.arange(number) - 0.5 * (number - 1)
    ) * slot_width

    return (
        dict(zip(inj_ids, offsets)),
        0.82 * slot_width,
    )


def entry_parameter_data(entry, parameter):
    if parameter not in entry["parameters"]:
        return None

    index = entry["parameters"].index(parameter)

    return {
        "normalized_plot": entry["normalized_plot"][index],
        "normalized_full": entry["normalized_full"][index],
        "cv_point": float(entry["cv_points"][index]),
        "pe_median": float(entry["pe_medians"][index]),
    }


def draw_one_grouped_violin(
    ax,
    values,
    position,
    width,
    color,
):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size < 2:
        return None

    violin = ax.violinplot(
        [values],
        positions=[position],
        widths=width,
        showextrema=False,
        quantiles=[list(VIOLIN_QUANTILES)],
    )

    body = violin["bodies"][0]
    body.set_facecolor(color)
    body.set_edgecolor(color)
    body.set_alpha(0.62)
    body.set_linewidth(0.8)

    if "cquantiles" in violin:
        violin["cquantiles"].set_color(color)
        violin["cquantiles"].set_linewidth(1.0)

    return violin


def make_grouped_violin_figure(
    entries,
    parameters,
    title,
):
    entries_by_inj = {
        entry["inj_id"]: entry
        for entry in entries
    }

    inj_ids_here = [
        inj_id
        for inj_id in available_inj_ids
        if inj_id in entries_by_inj
    ]

    offsets, violin_width = grouped_offsets(
        inj_ids_here
    )

    base_positions = np.arange(
        1,
        len(parameters) + 1,
        dtype=float,
    )

    all_sample_lists = []
    all_cv_points = []

    fig_width = max(
        13.0,
        1.15 * len(parameters),
    )
    fig, ax = plt.subplots(
        figsize=(fig_width, 7.0)
    )

    for inj_id in inj_ids_here:
        entry = entries_by_inj[inj_id]
        color = INJ_COLORS[inj_id]

        for parameter_index, parameter in enumerate(
            parameters
        ):
            data = entry_parameter_data(
                entry,
                parameter,
            )

            if data is None:
                continue

            position = (
                base_positions[parameter_index]
                + offsets[inj_id]
            )

            draw_one_grouped_violin(
                ax=ax,
                values=data["normalized_plot"],
                position=position,
                width=violin_width,
                color=color,
            )

            ax.scatter(
                position,
                data["cv_point"],
                marker="o",
                s=44,
                facecolor=color,
                edgecolor="black",
                linewidth=0.55,
                zorder=6,
            )

            if SHOW_PE_MEDIAN:
                ax.scatter(
                    position,
                    data["pe_median"],
                    marker="_",
                    s=130,
                    linewidths=1.5,
                    color="black",
                    zorder=7,
                )

            all_sample_lists.append(
                data["normalized_full"]
            )
            all_cv_points.append(
                data["cv_point"]
            )

    for position in base_positions:
        ax.axvline(
            position,
            color="black",
            linewidth=0.45,
            alpha=0.55,
            zorder=0,
        )

    ax.axhline(
        0.0,
        color="black",
        linewidth=1.0,
    )

    if SHOW_PLUS_MINUS_ONE:
        for level in (-1.0, 1.0):
            ax.axhline(
                level,
                color="black",
                linestyle="--",
                linewidth=0.7,
                alpha=0.60,
            )

    limits = (
        Y_LIMITS
        if Y_LIMITS is not None
        else automatic_y_limits(
            all_sample_lists,
            all_cv_points,
        )
    )
    ax.set_ylim(limits)

    ax.set_xlim(
        0.45,
        len(parameters) + 0.55,
    )
    ax.set_xticks(base_positions)
    ax.set_xticklabels(
        [
            parameter_label(parameter)
            for parameter in parameters
        ],
        fontsize=11,
    )

    ax.set_ylabel(
        r"$(\theta-\theta_{\rm ref})/"
        r"\sigma_{\rm CV}$"
    )
    ax.set_title(title)

    injection_handles = [
        Patch(
            facecolor=INJ_COLORS[inj_id],
            edgecolor=INJ_COLORS[inj_id],
            alpha=0.62,
            label=injection_label(inj_id),
        )
        for inj_id in inj_ids_here
    ]

    statistic_handles = [
        Line2D(
            [],
            [],
            marker="o",
            markersize=7,
            markerfacecolor="white",
            markeredgecolor="black",
            linestyle="None",
            label="CV bias",
        )
    ]

    if SHOW_PE_MEDIAN:
        statistic_handles.append(
            Line2D(
                [],
                [],
                marker="_",
                markersize=13,
                markeredgewidth=1.5,
                color="black",
                linestyle="None",
                label="PE median",
            )
        )

    first_legend = ax.legend(
        handles=injection_handles,
        title="Posterior violin",
        loc="upper left",
        ncol=min(3, len(injection_handles)),
    )
    ax.add_artist(first_legend)

    ax.legend(
        handles=statistic_handles,
        loc="upper right",
        ncol=len(statistic_handles),
    )

    ax.grid(False)
    fig.tight_layout()

    return fig, ax

## 11. Create one grouped plot per FTI recovery

In [ ]:
entries_by_fti = defaultdict(list)

for plot_case in plot_cases:
    entries_by_fti[
        plot_case["active_fti"]
    ].append(plot_case)

active_fti_order = ordered_active_fti_parameters(
    entries_by_fti.keys()
)

created_rows = []

for active_fti in active_fti_order:
    entries = entries_by_fti[active_fti]
    parameters = combined_parameter_order(entries)

    figure, axis = make_grouped_violin_figure(
        entries=entries,
        parameters=parameters,
        title=(
            f"{parameter_label(active_fti)} recovery: "
            f"comparison of injection IDs\n"
            f"reference = {CENTER_MODE}"
        ),
    )

    png_path = (
        PER_FTI_DIR
        / (
            f"{active_fti}_grouped_posterior_violin_"
            f"cv_bias_inj_ids_{inj_slug}.png"
        )
    )
    pdf_path = png_path.with_suffix(".pdf")

    figure.savefig(
        png_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
    )
    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    created_rows.append({
        "active_fti_run": active_fti,
        "inj_ids_present": ",".join(
            str(entry["inj_id"])
            for entry in sorted(
                entries,
                key=lambda item: item["inj_id"],
            )
        ),
        "number_of_parameters": len(parameters),
        "png_file": str(png_path),
        "pdf_file": str(pdf_path),
    })

created_table = pd.DataFrame(created_rows)

created_path = (
    TABLE_DIR
    / f"created_grouped_plots_inj_ids_{inj_slug}.csv"
)
created_table.to_csv(created_path, index=False)

display(created_table)
print("Saved plot index:", created_path)

## 12. Save all grouped recovery panels in one PDF

In [ ]:
all_runs_pdf = (
    OUTPUT_ROOT
    / (
        f"all_grouped_fti_posterior_violin_"
        f"cv_bias_inj_ids_{inj_slug}.pdf"
    )
)

with PdfPages(all_runs_pdf) as pdf:
    for active_fti in active_fti_order:
        entries = entries_by_fti[active_fti]
        parameters = combined_parameter_order(entries)

        figure, axis = make_grouped_violin_figure(
            entries=entries,
            parameters=parameters,
            title=(
                f"{parameter_label(active_fti)} recovery: "
                f"comparison of injection IDs\n"
                f"reference = {CENTER_MODE}"
            ),
        )

        pdf.savefig(
            figure,
            bbox_inches="tight",
        )
        plt.close(figure)

print("Saved multipage PDF:", all_runs_pdf)

## 13. Grouped active-FTI-only summary plot

In [ ]:
active_entry_lookup = {}

for plot_case in plot_cases:
    active_fti = plot_case["active_fti"]

    if active_fti not in plot_case["parameters"]:
        continue

    index = plot_case["parameters"].index(active_fti)

    active_entry_lookup[
        (plot_case["inj_id"], active_fti)
    ] = {
        "normalized_full": (
            plot_case["normalized_full"][index]
        ),
        "normalized_plot": (
            plot_case["normalized_plot"][index]
        ),
        "cv_point": float(
            plot_case["cv_points"][index]
        ),
        "pe_median": float(
            plot_case["pe_medians"][index]
        ),
    }

active_parameters = ordered_active_fti_parameters(
    {
        active_fti
        for _, active_fti in active_entry_lookup
    }
)

if not active_parameters:
    raise ValueError(
        "No active-FTI posterior entries were available."
    )

offsets, violin_width = grouped_offsets(
    available_inj_ids
)
base_positions = np.arange(
    1,
    len(active_parameters) + 1,
    dtype=float,
)

all_active_samples = []
all_active_cv = []

fig_width = max(
    13.0,
    1.15 * len(active_parameters),
)
figure, axis = plt.subplots(
    figsize=(fig_width, 7.0)
)

for inj_id in available_inj_ids:
    color = INJ_COLORS[inj_id]

    for parameter_index, active_fti in enumerate(
        active_parameters
    ):
        data = active_entry_lookup.get(
            (inj_id, active_fti)
        )

        if data is None:
            continue

        position = (
            base_positions[parameter_index]
            + offsets[inj_id]
        )

        draw_one_grouped_violin(
            ax=axis,
            values=data["normalized_plot"],
            position=position,
            width=violin_width,
            color=color,
        )

        axis.scatter(
            position,
            data["cv_point"],
            marker="o",
            s=44,
            facecolor=color,
            edgecolor="black",
            linewidth=0.55,
            zorder=6,
        )

        if SHOW_PE_MEDIAN:
            axis.scatter(
                position,
                data["pe_median"],
                marker="_",
                s=130,
                linewidths=1.5,
                color="black",
                zorder=7,
            )

        all_active_samples.append(
            data["normalized_full"]
        )
        all_active_cv.append(
            data["cv_point"]
        )

for position in base_positions:
    axis.axvline(
        position,
        color="black",
        linewidth=0.45,
        alpha=0.55,
        zorder=0,
    )

axis.axhline(
    0.0,
    color="black",
    linewidth=1.0,
)

if SHOW_PLUS_MINUS_ONE:
    for level in (-1.0, 1.0):
        axis.axhline(
            level,
            color="black",
            linestyle="--",
            linewidth=0.7,
            alpha=0.60,
        )

limits = (
    Y_LIMITS
    if Y_LIMITS is not None
    else automatic_y_limits(
        all_active_samples,
        all_active_cv,
    )
)
axis.set_ylim(limits)

axis.set_xlim(
    0.45,
    len(active_parameters) + 0.55,
)
axis.set_xticks(base_positions)
axis.set_xticklabels(
    [
        parameter_label(parameter)
        for parameter in active_parameters
    ],
    fontsize=11,
)

axis.set_ylabel(
    r"$(\theta-\theta_{\rm ref})/"
    r"\sigma_{\rm CV}$"
)
axis.set_title(
    "Active FTI posterior distributions and CV biases\n"
    f"inj_ids = {available_inj_ids}; reference = {CENTER_MODE}"
)

injection_handles = [
    Patch(
        facecolor=INJ_COLORS[inj_id],
        edgecolor=INJ_COLORS[inj_id],
        alpha=0.62,
        label=injection_label(inj_id),
    )
    for inj_id in available_inj_ids
]

statistic_handles = [
    Line2D(
        [],
        [],
        marker="o",
        markersize=7,
        markerfacecolor="white",
        markeredgecolor="black",
        linestyle="None",
        label="CV bias",
    )
]

if SHOW_PE_MEDIAN:
    statistic_handles.append(
        Line2D(
            [],
            [],
            marker="_",
            markersize=13,
            markeredgewidth=1.5,
            color="black",
            linestyle="None",
            label="PE median",
        )
    )

first_legend = axis.legend(
    handles=injection_handles,
    title="Posterior violin",
    loc="upper left",
    ncol=min(3, len(injection_handles)),
)
axis.add_artist(first_legend)

axis.legend(
    handles=statistic_handles,
    loc="upper right",
    ncol=len(statistic_handles),
)

axis.grid(False)
figure.tight_layout()

active_png = (
    OUTPUT_ROOT
    / (
        f"active_fti_grouped_posterior_violin_"
        f"cv_bias_inj_ids_{inj_slug}.png"
    )
)
active_pdf = active_png.with_suffix(".pdf")

figure.savefig(
    active_png,
    dpi=SAVE_DPI,
    bbox_inches="tight",
)
figure.savefig(
    active_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:", active_png)
print("Saved:", active_pdf)

## 14. Compare PE medians and CV biases across injections

In [ ]:
active_summary = summary_table.loc[
    summary_table["parameter"]
    == summary_table["active_fti_run"]
].copy()

active_summary = active_summary.sort_values(
    ["active_fti_run", "inj_id"]
).reset_index(drop=True)

active_summary_path = (
    TABLE_DIR
    / f"active_fti_summary_inj_ids_{inj_slug}.csv"
)
active_summary.to_csv(
    active_summary_path,
    index=False,
)

display(
    active_summary[
        [
            "inj_id",
            "active_fti_run",
            "CV_bias_over_sigma",
            "PE_median_over_sigma_CV",
            "PE_q05_over_sigma_CV",
            "PE_q95_over_sigma_CV",
            "PE_median_minus_CV",
        ]
    ]
)

print("Saved:", active_summary_path)

## 15. Largest PE-median versus CV differences

In [ ]:
largest_differences = (
    summary_table.assign(
        absolute_PE_median_minus_CV=lambda frame: (
            frame["PE_median_minus_CV"].abs()
        )
    )
    .sort_values(
        "absolute_PE_median_minus_CV",
        ascending=False,
    )
    .reset_index(drop=True)
)

largest_path = (
    TABLE_DIR
    / (
        f"largest_pe_median_minus_cv_"
        f"inj_ids_{inj_slug}.csv"
    )
)
largest_differences.to_csv(
    largest_path,
    index=False,
)

display(largest_differences.head(50))
print("Saved:", largest_path)

## 16. List files created

In [ ]:
created_files = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

print("This notebook writes only under:")
print(OUTPUT_ROOT)
print()
print("Number of files:", len(created_files))

for path in created_files:
    print(path)